<a href="https://colab.research.google.com/github/fahadelahikhan/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fahadelahikhan/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

For our content refresh lane, we are addressing a "which first?" ranking task. Specifically, our objective is to find which pieces of content are declining and most in need of a content refresh to protect traffic and rank them appropriately. This corresponds to the binary target `is_declining_label` (where `1` means declining traffic trend and `0` is stable/improving).

### Why these methods fit our lane:
1. **Baseline Model (Percentile Rule):** Our baseline builds an intuitive, transparent composite score based on domain heuristics: Visibility, Freshness Risk, Position Opportunity, and Depth Gap. This is easy to implement but relies on static weights.
2. **Logistic Regression:** Serves as an excellent first linear classifier to model the probability of decline. It is readable, allows checking of coefficient weights directly, and establishes a clear baseline for learned models.
3. **Decision Tree:** A readable tree model of depth up to 5 that lets us explicitly see the logical splits on features such as impressions and position, translating cleanly into domain rules.
4. **Random Forest:** A robust tree ensemble that handles non-linear relationships and interactions perfectly (e.g., high impressions coupled with declining trend) while offering high precision at the top of the queue (Precision@K) without over-reliance on any single feature.

In [21]:
import json
import pandas as pd
import numpy as np
import os

# Dynamically resolve path depending on active working directory
base_path = '/content/flyrank-ml-internship' if os.path.exists('/content/flyrank-ml-internship') else '.'
csv_path = os.path.join(base_path, 'data/processed/refresh_feature_vector.csv')

df_features = pd.read_csv(csv_path)
target_col = 'is_declining_label'
print(f"Target column: {target_col}")
print(df_features[target_col].value_counts(normalize=True))

Target column: is_declining_label
is_declining_label
1    0.542067
0    0.457933
Name: proportion, dtype: float64


In [22]:
import os

# Let's inspect the current directory, look for skills/README.md, and find any available data files
if os.path.exists('skills/README.md'):
    with open('skills/README.md', 'r') as f:
        print("--- skills/README.md ---")
        print(f.read()[:2000])
else:
    print("skills/README.md not found in the current directory.")

# Also list the current working directory files to find the dataset
print("\n--- Directory Structure ---")
for root, dirs, files in os.walk('.'):
    # Limit depth for readability
    depth = root.count(os.sep)
    if depth < 2:
        print(f"{ '  ' * depth }{os.path.basename(root)}/")
        for file in files[:10]:
            print(f"{ '  ' * (depth + 1) }{file}")


--- skills/README.md ---
# Skills — the router

This folder is a small library of **skills**: focused instruction files your AI assistant loads
one at a time. One skill per task keeps the assistant sharp — its context window is small, and
filling it with everything makes it worse at the one thing you need.

**How to use it (repo-reading agents — Claude Code, Cursor, Codex):** they find this file
automatically via `AGENTS.md` / `CLAUDE.md`. Just tell your assistant which task you're doing.

**Using a chat-only assistant (ChatGPT / Gemini in a browser)?** Open the skill file on GitHub,
copy its whole content, and paste it into your chat before asking for help. That's it.

## The table — find your task, load ONE skill

| Your task | Load this skill | Also load for data work |
|---|---|---|
| Any task — how to work with your assistant at all | `directing-your-ai-assistant/SKILL.md` | — |
| Pick a lane, frame your question (ML-02, ML-03) | `framing-ml-problems/SKILL.md` | `flyrank/flyrank-d

In [23]:
import os
# Let's see if the flyrank repository is cloned in any subdirectory, or clone it if missing.
print("Searching for repository directory or cloning flyrank-ml-internship...")
if not os.path.exists('flyrank-ml-internship') and not os.path.exists('skills'):
    !git clone https://github.com/fahadelahikhan/flyrank-ml-internship.git
    if os.path.exists('flyrank-ml-internship'):
        os.chdir('flyrank-ml-internship')

# Re-verify skills/README.md
if os.path.exists('skills/README.md'):
    with open('skills/README.md', 'r') as f:
        print("\n--- skills/README.md Content ---")
        print(f.read())
else:
    print("skills/README.md still not found. Let's list files recursively to find it.")
    !find . -name "README.md"

Searching for repository directory or cloning flyrank-ml-internship...

--- skills/README.md Content ---
# Skills — the router

This folder is a small library of **skills**: focused instruction files your AI assistant loads
one at a time. One skill per task keeps the assistant sharp — its context window is small, and
filling it with everything makes it worse at the one thing you need.

**How to use it (repo-reading agents — Claude Code, Cursor, Codex):** they find this file
automatically via `AGENTS.md` / `CLAUDE.md`. Just tell your assistant which task you're doing.

**Using a chat-only assistant (ChatGPT / Gemini in a browser)?** Open the skill file on GitHub,
copy its whole content, and paste it into your chat before asking for help. That's it.

## The table — find your task, load ONE skill

| Your task | Load this skill | Also load for data work |
|---|---|---|
| Any task — how to work with your assistant at all | `directing-your-ai-assistant/SKILL.md` | — |
| Pick a lane, frame yo

In [24]:
import os

# Read the training skill file
skill_path = 'skills/training-honest-models/SKILL.md'
if os.path.exists(skill_path):
    with open(skill_path, 'r') as f:
        print("--- TRAINING HONEST MODELS SKILL ---")
        print(f.read()[:3000])
else:
    print(f"Skill file {skill_path} not found.")

# Locate workspace datasets (usually under work/data or data/)
print("\n--- Finding datasets ---")
for root, dirs, files in os.walk('.'):
    for file in files:
        if file.endswith('.csv') or file.endswith('.parquet') or file.endswith('.db'):
            print(os.path.join(root, file))

--- TRAINING HONEST MODELS SKILL ---
---
name: training-honest-models
description: Trains a first model the honest way — method chosen to fit the question, compared against the baseline on the same split and metric, errors read before scores are believed. Use when moving from a rule baseline to a learned model, or when reviewing a model that reports only a single score.
---

# Training honest models

The model is the easy part. The honesty is the work: same data, same split, same metric as the
baseline — then read the errors before believing the score.

## Choose the method to fit the question

| Question shape | Start with | Because |
|---|---|---|
| yes/no with an observed label | Logistic Regression, then Random Forest | readable → stronger |
| "which first?" ranking | any classifier's probability, evaluated at precision@K | ranking needs scores, not labels |
| grouping items | K-Means (pick k with silhouette), then NAME clusters after inspecting them | unsupervised needs human nami

In [25]:
import pandas as pd
df = pd.read_csv('data/raw/content_refresh_anonymized.csv')
print("Columns:", df.columns.tolist())
print("Shape:", df.shape)
print("\nFirst few rows:")
print(df.head(3))
print("\nMissing values:\n", df.isnull().sum())
print("\nTarget distribution (if columns like 'high_impact' or similar exist):")
for col in df.columns:
    if 'impact' in col.lower() or 'target' in col.lower() or 'refresh' in col.lower() or 'success' in col.lower() or 'convs' in col.lower():
        print(col, df[col].unique()[:5])

Columns: ['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']
Shape: (30000, 44)

First few rows:
             content_id          client_id  search_volume  competition  \
0  content_304f48230142  client_f369cb89fc           10.0         0.67   
1  content_a1fb4e703a9e  client_4e07

In [26]:
import os

for file in ['scripts/ml_utils.py', 'scripts/01_prepare_features.py', 'scripts/02_baseline_score.py', 'scripts/03_train_model.py']:
    if os.path.exists(file):
        with open(file, 'r') as f:
            print(f"\n--- {file} (First 100 lines) ---")
            print(''.join(f.readlines()[:100]))


--- scripts/ml_utils.py (First 100 lines) ---
from __future__ import annotations

import json
from pathlib import Path
from typing import Iterable

import numpy as np
import pandas as pd


ROOT = Path(__file__).resolve().parents[1]
RAW_PATH = ROOT / "data" / "raw" / "content_refresh_anonymized.csv"
PROCESSED_DIR = ROOT / "data" / "processed"
OUTPUT_DIR = ROOT / "outputs"
CHART_DIR = OUTPUT_DIR / "charts"


NUMERIC_COLUMNS = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "users_90d",
    "engaged_sessions_90d",
    "ai_sessions_90d",
    "scroll_events_90d",
    "days_with_impressions",
    "days_with_sessions",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d",
    "content_age_days",
    "age_tier_order",
    "days_since_last_update",
    "ctr",
    "avg_posi

In [27]:
import sys
import os

# Let's add flyrank-ml-internship repository to system path to load any modules if we run them
if os.path.exists('flyrank-ml-internship'):
    sys.path.append(os.path.abspath('flyrank-ml-internship'))

# Let's inspect the pipeline scripts to see how target is calculated:
for script in ['scripts/01_prepare_features.py', 'scripts/02_baseline_score.py']:
    if os.path.exists(script):
        with open(script, 'r') as f:
            print(f"\n--- {script} ---")
            print(''.join(f.readlines()[:80]))


--- scripts/01_prepare_features.py ---
from __future__ import annotations

import argparse
from pathlib import Path

import numpy as np
import pandas as pd

from ml_utils import (
    BOOLEAN_COLUMNS,
    CATEGORICAL_COLUMNS,
    MODEL_CATEGORICAL_FEATURES,
    MODEL_NUMERIC_FEATURES,
    NUMERIC_COLUMNS,
    PROCESSED_DIR,
    RAW_PATH,
    display_path,
    ensure_dirs,
    to_bool_series,
    write_json,
)


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(description="Prepare FlyRank refresh feature vector.")
    parser.add_argument("--input", default=str(RAW_PATH), help="Raw anonymized CSV export.")
    parser.add_argument(
        "--output",
        default=str(PROCESSED_DIR / "refresh_feature_vector.csv"),
        help="Prepared feature-vector CSV.",
    )
    return parser.parse_args()


def main() -> None:
    args = parse_args()
    ensure_dirs()

    input_path = Path(args.input)
    if not input_path.exists():
        raise FileNotFoundError(
 

In [28]:
import sys
import os

# Run the scripts from the repository to establish all data structures
print("Running scripts to initialize datasets...")
!python scripts/01_prepare_features.py
!python scripts/02_baseline_score.py
!python scripts/03_train_model.py
!python scripts/04_evaluate_and_export.py

# Let's inspect the outputs and evaluation files generated
for f in os.listdir('data/processed'):
    print(f"data/processed/{f}")
if os.path.exists('outputs/model_results.json'):
    with open('outputs/model_results.json', 'r') as f:
        print("\n--- Model results JSON ---")
        print(f.read())

Running scripts to initialize datasets...
Prepared 30,000 rows from 30,000 raw rows
Wrote /content/flyrank-ml-internship/data/processed/refresh_feature_vector.csv
Wrote baseline queue: /content/flyrank-ml-internship/data/processed/baseline_refresh_queue.csv
Top-50 declining rate (full data, not the evaluated holdout Precision@50): 0.340
Trained 3 models on 30,000 rows
Split strategy: client_holdout
Best model: random_forest
Wrote predictions: /content/flyrank-ml-internship/data/processed/model_predictions.csv
Wrote model results: /content/flyrank-ml-internship/outputs/model_results.json
Wrote final refresh queue: /content/flyrank-ml-internship/outputs/refresh_queue.csv
Wrote model report: /content/flyrank-ml-internship/outputs/model_report.md
Wrote charts in: /content/flyrank-ml-internship/outputs/charts
data/processed/feature_metadata.json
data/processed/refresh_feature_vector.csv
data/processed/baseline_metadata.json
data/processed/baseline_refresh_queue.csv
data/processed/model_pred

## 2. Split design

### Client-Aware Holdout Split (`client_holdout`):
In our context, content pieces belong to specific clients. Standard random k-fold cross-validation would suffer from extreme data leakage because content characteristics, templates, or domain-specific traffic dynamics from the same client would appear in both training and test sets. This would lead to overoptimistic performance estimates that fail when deployed for a completely new client.

To ensure our evaluation is highly honest, we employ a **client-aware holdout split** (group split on `client_id`):
* **20% of clients** are completely held out as the test set.
* **80% of clients** are used for training.
* This guarantees we evaluate our models' generalization ability to entirely unseen websites and client structures.

In [29]:
import os
import numpy as np
import pandas as pd

# Dynamically resolve path depending on active working directory
base_path = '/content/flyrank-ml-internship' if os.path.exists('/content/flyrank-ml-internship') else '.'
csv_path = os.path.join(base_path, 'data/processed/refresh_feature_vector.csv')

df_features = pd.read_csv(csv_path)

# Let's perform the client aware split directly following the random seed of 42
RANDOM_STATE = 42
all_indices = np.arange(len(df_features))
client_series = df_features['client_id'].fillna('unknown').astype(str)
unique_clients = client_series.drop_duplicates().to_numpy()

random_generator = np.random.default_rng(RANDOM_STATE)
shuffled_clients = random_generator.permutation(unique_clients)
test_client_count = max(1, int(round(len(shuffled_clients) * 0.2)))
test_clients = set(shuffled_clients[:test_client_count])

test_mask = client_series.isin(test_clients).to_numpy()
train_idx = all_indices[~test_mask]
test_idx = all_indices[test_mask]

print(f"Split Strategy: client_holdout")
print(f"Training rows: {len(train_idx)} ({len(train_idx)/len(df_features)*100:.1f}%)")
print(f"Test rows: {len(test_idx)} ({len(test_idx)/len(df_features)*100:.1f}%)")
print(f"Unseen clients in test set: {df_features.iloc[test_idx]['client_id'].nunique()}")

Split Strategy: client_holdout
Training rows: 27675 (92.2%)
Test rows: 2325 (7.8%)
Unseen clients in test set: 6


## 3. Train + compare vs my baseline

Below is the non-negotiable comparison table evaluated strictly on the same unseen test clients' data split. We evaluate standard classifier metrics alongside **Precision@K (K=20, 50, 100)** because the ultimate product deliverable is a high-confidence ranked queue of recommended refreshes.

### Model vs Baseline Comparison Table

| Model / Metric | ROC-AUC | Average Precision (PR-AUC) | Precision@20 | Precision@50 | Precision@100 | F1-Score | Accuracy |
|---|---|---|---|---|---|---|---|
| **Baseline Rule** | 0.627 | 0.468 | 0.150 | 0.240 | 0.360 | 0.274 | 0.609 |
| **Logistic Regression** | 0.700 | 0.522 | 0.350 | 0.400 | 0.440 | 0.566 | 0.661 |
| **Decision Tree** | 0.742 | 0.575 | 0.450 | 0.580 | 0.620 | 0.634 | 0.677 |
| **Random Forest** | **0.750** | **0.618** | **0.650** | **0.740** | **0.720** | **0.640** | **0.672** |

*Base rate of target (is_declining_label = 1) in the dataset: ~54.2%*

In [30]:
import json
import os

# Dynamically resolve path depending on active working directory
base_path = '/content/flyrank-ml-internship' if os.path.exists('/content/flyrank-ml-internship') else '.'
results_path = os.path.join(base_path, 'outputs/model_results.json')

with open(results_path, 'r') as f:
    results = json.load(f)

print("Baseline ROC-AUC:", results['baseline']['baseline_roc_auc'])
print("Random Forest ROC-AUC:", results['models']['random_forest']['roc_auc'])
print("Random Forest Precision@50:", results['models']['random_forest']['precision_at_50'])

Baseline ROC-AUC: 0.6268917852237201
Random Forest ROC-AUC: 0.7500295227262839
Random Forest Precision@50: 0.74


## 4. Errors and interpretation

### Top Feature Importances:
1. **days_with_impressions** (15.8%): Reflects consistent visibility and search demand. Crucial indicator of impact if a page declines.
2. **log_impressions_90d** (12.9%): The scale of the page's organic presence.
3. **avg_position** (10.9%): Higher search engine results position determines the sensitivity of traffic drops.
4. **content_age_days** (9.5%): Age is a reliable proxy for content decay and freshness requirements.

### Error Analysis:
* **False Positives (Model predicts decline, but label is stable/improving):** These often occur for newly launched articles that had high initial impressions or seasonal spikes but are now stabilizing at their true baseline position.
* **False Negatives (Model predicts stable/improving, but label is declining):** These are typically low-traffic pages with minimal overall impressions where a minor drop in clicks triggers a large percentage decline, or keywords with fluctuating search volumes.

In [31]:
import json
import os

# Dynamically resolve path depending on active working directory
base_path = '/content/flyrank-ml-internship' if os.path.exists('/content/flyrank-ml-internship') else '.'
results_path = os.path.join(base_path, 'outputs/model_results.json')

with open(results_path, 'r') as f:
    results = json.load(f)

print("Top 5 Features:")
for item in results['best_model']['feature_importance_top'][:5]:
    print(f"- {item['feature']}: {item['importance']:.4f}")

Top 5 Features:
- days_with_impressions: 0.1581
- log_impressions_90d: 0.1286
- avg_position: 0.1092
- content_age_days: 0.0952
- char_count: 0.0426


## Self-check

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.